In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

from scipy.ndimage import gaussian_filter
from scipy.stats import zscore

from scipy.cluster import hierarchy
from scipy.spatial.distance import pdist
import matplotlib.pyplot as plt

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc'
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))

In [ ]:
tmp_order = ['Start', 'Sendai', 'Delayed', 'Fail1', 'Inter1',  'Inter2', 'Fail2', 'IPS']
samples = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

In [ ]:
def process_bed(s):
    pmds_mc = pd.read_csv(f'{projdir}/bed/pmds/rna_cluster_donor/{s}.start_pmds_CGN-Both.full.bed.gz', sep='\t', header=None)
    pmds_mc.index = [f'{x}_{y}' for x,y in zip(pmds_mc[0], pmds_mc[1])]
    return pmds_mc

with ProcessPoolExecutor() as executor:
    results = list(tqdm(executor.map(process_bed, samples), total=len(samples)))

In [ ]:
results[0].head()

In [ ]:
mc_results = [x[3] / x[4] for x in results]
pmds_df = pd.concat(mc_results, axis=1)
pmds_df.columns = samples

for s in samples:
    pmds_df[s].fillna(pmds_df[s].mean(), inplace=True)
    
pmds_df = pmds_df[~pmds_df.index.duplicated()]
pmds_df.shape

In [ ]:
pmds_df.head()

In [ ]:
pmds_df.shape

In [ ]:
corr_df = pmds_df.corr()
corr_df.head()

In [ ]:
ax = sns.heatmap(corr_df, vmin=0, vmax=1, cmap='coolwarm')
ax.set_xticklabels(ax.get_xticklabels(), fontsize=6)
ax.set_yticklabels([]);

### plot by donor

In [ ]:
tmp_samples = [f'{x}_{y}' for x,y in zip(np.tile(tmp_order, 4), np.repeat(donors, 8))]
tmp_df = corr_df.loc[tmp_samples,tmp_samples]

In [ ]:
tmp_df.head()

In [ ]:
fig, ax = plt.subplots(1, figsize=(8, 8))
ax = sns.heatmap(tmp_df, vmin=0.25, vmax=1, cmap='coolwarm')
ax.set_xticklabels(ax.get_xticklabels(), fontsize=9);
ax.set_yticklabels(ax.get_yticklabels(), fontsize=9);

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 12))
for i, ax in enumerate(axes.flatten()):
    d = donors[i]
    mask = [d in x for x in tmp_df.columns]
    donor_df = tmp_df.loc[:,mask]
    sns.heatmap(donor_df, vmin=0.25, vmax=1, cmap='coolwarm', ax=ax)
    ax.set_xticklabels(ax.get_xticklabels(), fontsize=6);
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=6);

In [ ]:
sns.clustermap(tmp_df, vmin=0.25, vmax=1, cmap='coolwarm',
               row_cluster=True, col_cluster=True)

In [ ]:
# compute linkage yourself
row_linkage = hierarchy.linkage(pdist(tmp_df), method='average')
col_linkage = hierarchy.linkage(pdist(tmp_df.T), method='average')

# get current leaf order and rotate branches
# find which node to rotate by inspecting the dendrogram
fig = plt.figure(figsize=(10, 8))
dend = hierarchy.dendrogram(row_linkage, labels=tmp_df.index)
plt.show()
# leaf order is in dend['leaves']
print(dend['leaves'])

In [ ]:
#                 'Inter2_C38', 'Fail2_C38', 'Inter2_C37', 'Inter2_C39', 'Inter2_C29', 'Fail2_C29',
#                 'Inter1_C38', 'Delayed_C38', 'Sendai_C38', 'Start_C38', 'Fail1_C38',
#                 'Inter1_C29', 'Start_C29', 'Sendai_C29', 'Delayed_C29', 'Fail1_C29',
#                 'Inter1_C37', 'Fail2_C37', 'Start_C37', 'Sendai_C37', 'Delayed_C37', 'Fail1_C37',
#                 'Fail2_C39', 'Fail1_C39', 'Start_C39', 'Sendai_C39', 'Delayed_C39', 'Inter1_C39']

In [ ]:
#                  4, 0, 1, 2, 3, 8, 9, 10, 11, 12, 14, 30, 27, 24, 25, 26, 28]

# # reindex df using new order

In [ ]:
from scipy.cluster import hierarchy
from scipy.spatial.distance import pdist
import numpy as np

# 1. compute linkage
row_linkage = hierarchy.linkage(pdist(tmp_df), method='average')

# 2. convert to ete3 tree
from ete3 import Tree

def linkage_to_ete(linkage_matrix, labels):
    n = len(labels)
    nodes = {i: Tree(name=labels[i]) for i in range(n)}
    for i, (left, right, dist, _) in enumerate(linkage_matrix):
        node = Tree()
        node.dist = dist
        node.add_child(nodes[int(left)])
        node.add_child(nodes[int(right)])
        nodes[n + i] = node
    return nodes[n + len(linkage_matrix) - 1]

labels = tmp_df.index.tolist()
tree = linkage_to_ete(row_linkage, labels)

In [ ]:
# first, see the full tree structure
print(tree.get_ascii(show_internal=True))

# more useful - see which leaves are in each subtree
for i, node in enumerate(tree.traverse()):
    if not node.is_leaf():
        leaves = node.get_leaf_names()
        if any('Inter2' in l or 'Fail2' in l for l in leaves):
            print(f"Node {i}: {leaves}")

In [ ]:
### rotate tree: put Inter2/Fail2 branch adjacent to IPS
# find Node2 (non-IPS child at root) and swap its children so Node6 (Inter2/Fail2) comes first
ips_leaves = {l for l in tmp_df.index if l.startswith('IPS_')}

node2 = None
for child in tree.children:
    if not ips_leaves.issubset(set(child.get_leaf_names())):
        node2 = child
        break

first_has_inter2 = any('Inter2' in l or 'Fail2' in l for l in node2.children[0].get_leaf_names())
if not first_has_inter2:
    node2.swap_children()
    print("rotated node2")

### rebuild scipy linkage from rotated ete3 tree
# same distances/tree shape, but left/right children reflect the rotation
# seaborn uses row_linkage to determine both the dendrogram and the row order

def ete_to_linkage(tree, labels):
    n = len(labels)
    label_to_idx = {l: i for i, l in enumerate(labels)}
    node_idx = {}
    rows = []
    new_idx = n
    for node in tree.traverse('postorder'):
        if node.is_leaf():
            node_idx[id(node)] = label_to_idx[node.name]
        else:
            left, right = node.children
            rows.append([node_idx[id(left)], node_idx[id(right)],
                         node.dist, len(node.get_leaf_names())])
            node_idx[id(node)] = new_idx
            new_idx += 1
    return np.array(rows)

rotated_linkage = ete_to_linkage(tree, labels)

# verify — first 8 leaves should be IPS then Inter2/Fail2
from scipy.cluster.hierarchy import dendrogram as _dend
_d = _dend(rotated_linkage, labels=labels, no_plot=True)
print([labels[i] for i in _d['leaves']][:10])

sns.clustermap(tmp_df, vmin=0.25, vmax=1, cmap='coolwarm',
               row_linkage=rotated_linkage, col_linkage=rotated_linkage)

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure2'
tmp_df.to_csv(f'{outdir}/rna_cluster_donor_pmd_mcg.csv', sep='\t', header=True, index=True)

In [ ]:
pd.DataFrame(rotated_linkage, columns=['left','right','dist','count']).to_csv(f'{outdir}/rna_cluster_donor_linkage.csv', index=False)                                                                     
pd.Series(labels).to_csv(f'{outdir}/rna_cluster_donor_linkage_labels.csv', index=False, header=False)    

In [ ]:
!date